In [ ]:
# تثبيت جميع المكتبات المطلوبة
!pip install opencv-python seaborn scikit-learn tensorflow pillow matplotlib pandas numpy -q

print("✅ تم تثبيت جميع المكتبات بنجاح!")

In [ ]:
# Cell 1: Imports & Global Settings

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# -----------------------
# Global Settings
# -----------------------
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# Cell 2: Load Metadata

BASE_DIR =r"data"   # عدّل إذا مسار الداتا مختلف
METADATA_PATH = os.path.join(BASE_DIR, "HAM10000_metadata.csv")

df = pd.read_csv(r"data/HAM10000_metadata.csv")

print("Dataset shape:", df.shape)
df.head()


In [ ]:
# Cell 3: Create Binary Labels

malignant_classes = ["mel", "bcc"]

df["binary_label"] = df["dx"].apply(
    lambda x: 1 if x in malignant_classes else 0
)

print(df["binary_label"].value_counts())
df[["image_id", "dx", "binary_label"]].head()


In [ ]:
# Cell 4: Train / Validation / Test Split

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["binary_label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["binary_label"],
    random_state=SEED
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


In [ ]:
# Cell 5: Image Paths

IMG_DIR_1 = os.path.join(BASE_DIR, "HAM10000_images_part_1")
IMG_DIR_2 = os.path.join(BASE_DIR, "HAM10000_images_part_2")

def get_image_path(image_id):
    p1 = os.path.join(IMG_DIR_1, image_id + ".jpg")
    p2 = os.path.join(IMG_DIR_2, image_id + ".jpg")
    return p1 if os.path.exists(p1) else p2

for split_df in [train_df, val_df, test_df]:
    split_df["image_path"] = split_df["image_id"].apply(get_image_path)

print("Train missing images:", train_df["image_path"].isnull().sum())
print("Validation missing images:", val_df["image_path"].isnull().sum())
print("Test missing images:", test_df["image_path"].isnull().sum())


In [ ]:
# Cell 6: Visual Check

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

benign_samples = train_df[train_df["binary_label"] == 0].sample(5)
malignant_samples = train_df[train_df["binary_label"] == 1].sample(5)

for i, (_, row) in enumerate(benign_samples.iterrows()):
    img = Image.open(row["image_path"])
    axes[0, i].imshow(img)
    axes[0, i].axis("off")
    axes[0, i].set_title("Benign")

for i, (_, row) in enumerate(malignant_samples.iterrows()):
    img = Image.open(row["image_path"])
    axes[1, i].imshow(img)
    axes[1, i].axis("off")
    axes[1, i].set_title("Malignant")

plt.tight_layout()
plt.show()


In [ ]:
# Cell 7: Dataset Class

class SkinDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df
        self.transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip() if augment else transforms.Lambda(lambda x: x),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        image = self.transform(image)
        label = torch.tensor(row["binary_label"], dtype=torch.float32)
        return image, label


In [ ]:
# Cell 8: DataLoaders

train_dataset = SkinDataset(train_df, augment=True)
val_dataset   = SkinDataset(val_df, augment=False)
test_dataset  = SkinDataset(test_df, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))


In [ ]:
# --- خلية جديدة لتحسين توازن البيانات ---
from torch.utils.data import WeightedRandomSampler

# حساب عدد العينات في كل فئة
class_counts = train_df['binary_label'].value_counts().to_dict()
# حساب الوزن لكل عينة (عكس التكرار)
weights = [1.0 / class_counts[label] for label in train_df['binary_label']]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

# إعادة تعريف train_loader مع الـ Sampler
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)

print("✅ تم تفعيل Weighted Sampler لموازنة البيانات بنجاح!")

In [ ]:
# --- الموديل والمحسن والمجدول المحدث (نسخة مصححة9) ---
import torch.nn as nn

# 1. تحميل موديل ResNet18
model = models.resnet18(pretrained=True)

# 2. تعديل الطبقة الأخيرة لتناسب التصنيف الثنائي
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)

# 3. تعريف دالة الخسارة والمحسن
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

# 4. إضافة Scheduler (تم حذف verbose=True لحل المشكلة)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

print("✅ تم إعداد النموذج والمحسن والمجدول بنجاح!")

In [ ]:
# --- حلقة التدريب المطورة (مع ميزة طباعة معدل التعلم10) ---
def train_model(model, train_loader, val_loader, epochs=10):
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # --- مرحلة التدريب ---
        model.train()
        train_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device).float()
            
            optimizer.zero_grad()
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # --- مرحلة التحقق (Validation) ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device).float()
                outputs = model(images).squeeze()
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        # الحصول على معدل التعلم الحالي للطباعة
        current_lr = optimizer.param_groups[0]['lr']
        
        # تحديث المجدول
        scheduler.step(avg_val_loss)

        print(f"Epoch [{epoch+1}/{epochs}] | LR: {current_lr:.6f} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # حفظ أفضل نسخة
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_skin_model.pth')
            print("⭐ تم حفظ أفضل نسخة للنموذج!")

# تشغيل التدريب
train_model(model, train_loader, val_loader, epochs=10)

In [ ]:
# Cell 11: Evaluation on Test Set

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images).squeeze()
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).int().cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=["Benign", "Malignant"]))


In [ ]:
# --- خلية التقييم الشامل ---
from sklearn.metrics import classification_report, confusion_matrix, recall_score

# تحميل أفضل نسخة تم حفظها قبل الاختبار
model.load_state_dict(torch.load('best_skin_model.pth'))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images).squeeze()
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).int().cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# حساب الحساسية (Sensitivity) وهي الأهم طبياً
sensitivity = recall_score(all_labels, all_preds)

print("Medical Evaluation Report:")
print("-" * 30)
print(classification_report(all_labels, all_preds, target_names=["Benign", "Malignant"]))
print(f"Final Sensitivity (Recall): {sensitivity:.2%}")